# AutoDL影像消融：I5：WAC+DEM+Slope+TPI

DeepLabV3+-ResNet50、seed42、physical batch4、80 epochs；只改变有效输入通道，不访问Test。


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, importlib.util, json, os, shutil, subprocess, sys

PROJECT_DIR = Path('/root/autodl-tmp/projects/lunar-linear/LTL-Net')
DATA_ROOT = Path('/root/autodl-tmp/datasets/dataset_v6_random811_overlap40')
OUTPUT_ROOT = Path('/root/autodl-tmp/outputs')
HF_CACHE_SOURCE = Path('/root/autodl-tmp/resnet50_imagenet_cache')
CONFIG_PATH = PROJECT_DIR / 'configs/v6_overlap40_deeplab_input_wac_dem_slope_tpi_batch4_seed42.json'

config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['channel_mode'] == 'wac_dem_slope_tpi'
assert config['module'] == 'deeplab' and config['seed'] == 42
assert config['batch_size'] == 4 and config['accum_steps'] == 1 and config['epochs'] == 80
assert config['automatic_test_evaluation'] is False
assert PROJECT_DIR.is_dir() and DATA_ROOT.is_dir()
required = [('rasterio', 'rasterio'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
expected_cache_hashes = {
    'config.json': '01bf2cf24eb29b405c28c159f46ceda92c098ab85868be88fb100967db47166e',
    'model.safetensors': 'df1aad85e18536504a4c8597118364e291ff3a9c4b56dd9b3a4900642e4c3a7c',
}
snapshot = Path('/root/.cache/huggingface/hub/models--smp-hub--resnet50.imagenet/snapshots/00cb74e366966d59cd9a35af57e618af9f88efe9')
snapshot.mkdir(parents=True, exist_ok=True)
for name, expected_hash in expected_cache_hashes.items():
    source = HF_CACHE_SOURCE / name; target = snapshot / name
    assert source.is_file(), f'离线ImageNet权重不存在: {source}'
    assert hashlib.sha256(source.read_bytes()).hexdigest() == expected_hash
    if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected_hash:
        shutil.copy2(source, target)
os.environ['HF_HUB_OFFLINE'] = '1'
commit = subprocess.check_output(['git', '-C', str(PROJECT_DIR.parent), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Git commit:', commit)
print('channel_mode:', config['channel_mode'])
print('输出:', OUTPUT_ROOT / f"result_{config['run_name']}")
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
command = [sys.executable, str(PROJECT_DIR / 'scripts/run_autodl_channel_ablation.py'),
           '--project-dir', str(PROJECT_DIR), '--config', str(CONFIG_PATH),
           '--data-dir', str(DATA_ROOT), '--output-dir', str(OUTPUT_ROOT)]
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'; env['HF_HUB_OFFLINE'] = '1'
print(' '.join(command), flush=True)
subprocess.check_call(command, cwd=PROJECT_DIR, env=env)


In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
metrics = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
assert metrics['test_evaluated'] is False
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('下载:', Path(str(result_dir) + '.zip'))
